%md
# Notebook 09 – ML Dashboard Code

### Why Dashboard Tables?

Healthcare users such as clinicians, hospital administrators, and analysts require interactive dashboards instead of raw database tables.

This notebook prepares business-ready datasets that power the Databricks SQL Dashboard by creating Key Performance Indicators (KPIs), summary tables, and machine learning reporting tables.

### Objective

The objective of this notebook is to generate dashboard-ready tables for monitoring hospital operations, patient outcomes, and machine learning predictions.

### Input Tables

This notebook reads the following Gold tables:

- gold.patient_cohorts
- gold.encounter_summaries
- gold.readmission_within_30days
- gold.readmission_predictions

### Output

This notebook produces dashboard-ready datasets used by Databricks SQL Dashboards, including:

- ADT Volume
- Readmission Rate
- Top Diagnoses
- Medication Usage
- Average Length of Stay
- Readmission Risk Distribution
- High-Risk Patients
- Readmission by Cohort
- Average Risk by Diagnosis

%md
### 1. Generate Readmission Risk Distribution

Prediction probabilities are grouped into risk categories.

The resulting dataset shows the distribution of Low-, Medium-, and High-Risk patients across the hospital population.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_readmission_risk_distribution AS
SELECT
CASE
    WHEN prediction = 1 THEN 'High Risk'
    ELSE 'Low Risk'
END AS risk_level,
COUNT(*) AS person_count
FROM gold.readmission_predictions
GROUP BY prediction;

### 2. Identify High-Risk Patients

Patients predicted to have a high probability of readmission are identified.

This dataset supports proactive clinical intervention and care management.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_high_risk_patients AS
SELECT
    person_id,
    age,
    primary_diagnosis,
    previous_admissions,
    prediction,
    CASE
        WHEN prediction = 1 THEN 'High Risk'
        ELSE 'Low Risk'
    END AS risk_level
FROM gold.readmission_predictions
WHERE prediction = 1;

### 3. Readmission by Cohort

Patients are grouped into cohorts based on demographic or clinical characteristics.

The resulting dataset allows hospitals to compare readmission trends across different patient populations.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_readmission_by_cohort AS
SELECT
c.cohort_name,
COUNT(*) AS high_risk_patient
FROM gold.readmission_predictions p
JOIN gold.patient_cohorts c
ON p.person_id = c.person_id
WHERE p.prediction = 1
GROUP BY c.cohort_name;


### 4. Average Risk by Diagnosis

The average predicted readmission probability is calculated for each diagnosis.

This helps identify medical conditions associated with higher readmission risk.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi_average_risk_by_diagnosis AS
SELECT
    primary_diagnosis,
    AVG(prediction) AS average_risk,
    COUNT(*) AS patient_count
FROM gold.readmission_predictions
GROUP BY primary_diagnosis
ORDER BY average_risk DESC;

### Dashboard Output

The generated datasets are consumed by the Databricks SQL Dashboard to provide interactive visualizations for:

- Hospital Administrators
- Clinicians
- Data Analysts
- Healthcare Executives

The dashboard provides real-time monitoring of operational KPIs and machine learning predictions to support clinical and business decision-making.